<a href="https://colab.research.google.com/github/Zahra-Mhdi/Deep-Learning-Exercises/blob/main/Session_6_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exercise:
Reach strong accuracy on CIFAR-10 by applying the techniques (data augmentation, optimizer selection, weight decay, schedulers, gradient clipping, AMP, Early Stopping).

In [5]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.optim.lr_scheduler as lr_sched
from torch.cuda.amp import autocast, GradScaler
from torch.nn.utils import clip_grad_norm_

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')

# Data augmentation
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

# CIFAR-10
train_full = datasets.CIFAR10(root='./data', train=True,
                              download=True, transform=train_transform)
test_ds = datasets.CIFAR10(root='./data', train=False,
                           download=True, transform=test_transform)

val_size = 5000
train_size = len(train_full) - val_size
train_ds, val_ds = random_split(train_full, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

# Small CNN
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 16x16

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 8x8

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 4x4
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = SmallCNN().to(device)

# Optimizer + weight decay
opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# LR scheduler
sched = lr_sched.StepLR(opt, step_size=20, gamma=0.1)

loss_fn = nn.CrossEntropyLoss()
scaler = GradScaler(enabled=use_amp)

# Train one epoch
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total = 0
    correct = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        opt.zero_grad()
        with autocast(enabled=use_amp):
            out = model(x)
            loss = loss_fn(out, y)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(opt)
        scaler.update()

        total_loss += loss.item() * x.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

# Eval
def eval_model(model, loader):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            with autocast(enabled=use_amp):
                out = model(x)
                loss = loss_fn(out, y)

            total_loss += loss.item() * x.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += x.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

# Train loop + early stopping
best_val_loss = float('inf')
patience = 5
counter = 0
best_state = None

for epoch in range(40):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = eval_model(model, val_loader)
    sched.step()

    print("Epoch", epoch+1,
          "train_loss", train_loss,
          "train_acc", train_acc,
          "val_loss", val_loss,
          "val_acc", val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_state = model.state_dict()
        torch.save(best_state, "best_cifar10.pt")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

# Test
test_loss, test_acc = eval_model(model, test_loader)
print("Test loss", test_loss)
print("Test acc", test_acc)


/tmp/ipython-input-1634239056.py:78: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/tmp/ipython-input-1634239056.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipython-input-1634239056.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 1 train_loss 1.720166848013136 train_acc 0.3708888888888889 val_loss 1.4760151123046874 val_acc 0.4634
Epoch 2 train_loss 1.3896781439251369 train_acc 0.49764444444444444 val_loss 1.3116754913330078 val_acc 0.522
Epoch 3 train_loss 1.2539342098024155 train_acc 0.5528666666666666 val_loss 1.1914649139404296 val_acc 0.571
Epoch 4 train_loss 1.1583039151933459 train_acc 0.5872666666666667 val_loss 1.115777735900879 val_acc 0.5922
Epoch 5 train_loss 1.0967597308794657 train_acc 0.6098666666666667 val_loss 1.0424550537109376 val_acc 0.6236
Epoch 6 train_loss 1.0391992611143324 train_acc 0.6324 val_loss 1.0511423828125 val_acc 0.6304
Epoch 7 train_loss 0.99360825881958 train_acc 0.6486888888888889 val_loss 1.016269175720215 val_acc 0.6354
Epoch 8 train_loss 0.9579563290490044 train_acc 0.6637111111111111 val_loss 0.9502462585449218 val_acc 0.6686
Epoch 9 train_loss 0.9227729566997952 train_acc 0.6764888888888889 val_loss 0.9021789703369141 val_acc 0.6772
Epoch 10 train_loss 0.890322343